# 02 — Evaluation

Reads all prediction CSVs from `results/predictions/` and computes the canonical AOD-retrieval metric panel
for every (split, model, target_transform) combination.

**Metrics:** RMSE, MAE, MBE, RMB, Pearson r, Spearman ρ, R², slope, intercept,
%within/above/below EE (Levy et al. 2013 Dark Target land envelope: EE = ±(0.05 + 0.15·τ))

**Also:** 95% bootstrap CIs (1,000 resamples), pairwise permutation tests (10,000 permutations),
per-quintile-bin MAE and MBE.

**Run after `01_train.ipynb` for both splits.**

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy import stats
from itertools import combinations

ROOT     = Path('D:/Regional-Scale-AOD-Estimation')
PRED_DIR = ROOT / 'modeling/results/predictions'
METRICS_DIR  = ROOT / 'modeling/results/metrics'
SIG_DIR      = ROOT / 'modeling/results/significance'
BIN_DIR      = ROOT / 'modeling/results/per_bin'
PLOT_DIR     = ROOT / 'modeling/results/plots'
SCATTER_DIR  = PLOT_DIR / 'scatter'

for d in [METRICS_DIR, SIG_DIR, BIN_DIR, PLOT_DIR, SCATTER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_BOOT = 1000
N_PERM = 10_000
SEED   = 42
rng    = np.random.default_rng(SEED)

print('Output dirs ready')

Output dirs ready


In [2]:
# Load all prediction CSVs
frames = []
for csv_path in sorted(PRED_DIR.glob('*/*.csv')):
    frames.append(pd.read_csv(csv_path))

all_preds = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(frames)} prediction files')
print(f'Splits:     {sorted(all_preds["split"].unique())}')
print(f'Models:     {sorted(all_preds["model"].unique())}')
print(f'Transforms: {sorted(all_preds["target_transform"].unique())}')
print(f'Total rows: {len(all_preds):,}')

Loaded 74 prediction files
Splits:     ['holdout_time', 'loso', 'stratified']
Models:     ['catboost', 'extratrees', 'gbm', 'lasso', 'lgbm', 'rf', 'ridge', 'svr', 'tabpfn_mi100', 'tabpfn_mi200', 'tabpfn_mi300', 'tabpfn_pca150', 'xgboost']
Transforms: ['log1p', 'raw']
Total rows: 161,542


In [3]:
def compute_metrics(y_true, y_pred):
    """Canonical AOD-retrieval metric panel."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    n = len(yt)

    residuals = yp - yt
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae  = np.mean(np.abs(residuals))
    mbe  = np.mean(residuals)
    rmb  = np.mean(yp) / np.mean(yt) if np.mean(yt) != 0 else np.nan

    pearson_r, _ = stats.pearsonr(yt, yp)
    spearman_r   = stats.spearmanr(yt, yp).statistic

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((yt - yt.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    slope, intercept, *_ = stats.linregress(yt, yp)

    # Levy et al. (2013) Dark Target land EE envelope: ±(0.05 + 0.15·AOD)
    EE = 0.05 + 0.15 * yt
    pct_within = np.mean(np.abs(residuals) <= EE) * 100
    pct_above  = np.mean(residuals > EE)  * 100
    pct_below  = np.mean(-residuals > EE) * 100

    return {
        'N': n, 'RMSE': rmse, 'MAE': mae, 'MBE': mbe, 'RMB': rmb,
        'Pearson_r': pearson_r, 'Spearman_r': spearman_r, 'R2': r2,
        'slope': slope, 'intercept': intercept,
        'pct_within_EE': pct_within, 'pct_above_EE': pct_above, 'pct_below_EE': pct_below,
    }


def bootstrap_ci(y_true, y_pred, n_boot=N_BOOT, rng=rng):
    """Bootstrap 95% CIs for all metrics. Returns {metric: (lo, hi)}."""
    n = len(y_true)
    boot_records = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_records.append(compute_metrics(y_true[idx], y_pred[idx]))
    boot_df = pd.DataFrame(boot_records)
    ci = {}
    for col in boot_df.columns:
        if col == 'N':
            continue
        lo = float(np.nanpercentile(boot_df[col], 2.5))
        hi = float(np.nanpercentile(boot_df[col], 97.5))
        ci[col] = (lo, hi)
    return ci


print('Metric functions defined')

Metric functions defined


In [4]:
metric_rows = []

groups = all_preds.groupby(['split', 'model', 'target_transform'])
total  = len(groups)
print(f'Computing metrics for {total} (split × model × transform) groups ...')

for i, ((split, model, transform), grp) in enumerate(groups):
    yt = grp['y_true'].values
    yp = grp['y_pred'].values

    m  = compute_metrics(yt, yp)
    ci = bootstrap_ci(yt, yp)

    row = {'split': split, 'model': model, 'transform': transform}
    row.update(m)
    for metric, (lo, hi) in ci.items():
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    metric_rows.append(row)

    if (i + 1) % 5 == 0 or (i + 1) == total:
        print(f'  {i+1}/{total} done')

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_DIR / 'all_metrics.csv', index=False)
print(f'\nSaved: {METRICS_DIR}/all_metrics.csv')

Computing metrics for 74 (split × model × transform) groups ...
  5/74 done
  10/74 done
  15/74 done
  20/74 done
  25/74 done
  30/74 done
  35/74 done
  40/74 done
  45/74 done
  50/74 done
  55/74 done
  60/74 done
  65/74 done
  70/74 done
  74/74 done

Saved: D:\Regional-Scale-AOD-Estimation\modeling\results\metrics/all_metrics.csv


In [5]:
# Display best-transform per model (pick row with lowest RMSE per split+model)
best = (
    metrics_df.sort_values('RMSE')
    .groupby(['split', 'model'], as_index=False)
    .first()
    .sort_values(['split', 'RMSE'])
)

display_cols = ['split', 'model', 'transform', 'N', 'RMSE', 'MAE', 'MBE',
                'Pearson_r', 'Spearman_r', 'R2', 'pct_within_EE']

for split_name, grp in best.groupby('split'):
    print(f'\n=== Split: {split_name} (best transform per model) ===')
    print(grp[display_cols].to_string(index=False))


=== Split: holdout_time (best transform per model) ===
       split         model transform   N     RMSE      MAE       MBE  Pearson_r  Spearman_r       R2  pct_within_EE
holdout_time  tabpfn_mi300       raw 834 0.034469 0.023211 -0.004108   0.967558    0.956137 0.934877      98.081535
holdout_time  tabpfn_mi200     log1p 834 0.034766 0.023552 -0.004868   0.967110    0.956364 0.933752      97.961631
holdout_time tabpfn_pca150     log1p 834 0.036321 0.023533 -0.000199   0.963195    0.951640 0.927689      97.841727
holdout_time           svr     log1p 834 0.036559 0.024854  0.000885   0.962989    0.943339 0.926740      97.002398
holdout_time  tabpfn_mi100     log1p 834 0.037181 0.025498  0.000256   0.961369    0.948159 0.924226      97.362110
holdout_time      catboost     log1p 834 0.042026 0.030203 -0.002145   0.951737    0.930286 0.903190      95.803357
holdout_time       xgboost       raw 834 0.042659 0.030257 -0.000189   0.951451    0.929594 0.900254      94.484412
holdout_time    

In [6]:
# Per-quintile-bin MAE and MBE
bin_rows = []
for (split, model, transform), grp in all_preds.groupby(['split', 'model', 'target_transform']):
    yt = grp['y_true'].values
    yp = grp['y_pred'].values
    quantiles = np.quantile(yt, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
    for b in range(5):
        lo, hi = quantiles[b], quantiles[b + 1]
        # Use strict upper bound (< hi) for bins 0-3 to avoid double-counting
        # boundary values; last bin uses >= lo only (captures the maximum).
        mask = (yt >= lo) & (yt < hi) if b < 4 else (yt >= lo)
        if mask.sum() == 0:
            continue
        bin_rows.append({
            'split': split, 'model': model, 'transform': transform,
            'bin_idx': b + 1,
            'aod_lo': round(lo, 3), 'aod_hi': round(hi, 3),
            'aod_mid': round((lo + hi) / 2, 3),
            'N': int(mask.sum()),
            'MAE': float(np.mean(np.abs(yp[mask] - yt[mask]))),
            'MBE': float(np.mean(yp[mask] - yt[mask])),
        })

bin_df = pd.DataFrame(bin_rows)
bin_df.to_csv(BIN_DIR / 'per_bin_mae_mbe.csv', index=False)
print(f'Saved: {BIN_DIR}/per_bin_mae_mbe.csv')
print(bin_df.head(10).to_string(index=False))

Saved: D:\Regional-Scale-AOD-Estimation\modeling\results\per_bin/per_bin_mae_mbe.csv
       split    model transform  bin_idx  aod_lo  aod_hi  aod_mid   N      MAE       MBE
holdout_time catboost     log1p        1   0.023   0.090    0.056 167 0.024056  0.018012
holdout_time catboost     log1p        2   0.090   0.133    0.111 167 0.019898  0.005062
holdout_time catboost     log1p        3   0.133   0.190    0.162 166 0.024105  0.006058
holdout_time catboost     log1p        4   0.190   0.272    0.231 167 0.030638 -0.006917
holdout_time catboost     log1p        5   0.272   0.989    0.631 167 0.052285 -0.032889
holdout_time catboost       raw        1   0.023   0.090    0.056 167 0.027034  0.023350
holdout_time catboost       raw        2   0.090   0.133    0.111 167 0.023567  0.009464
holdout_time catboost       raw        3   0.133   0.190    0.162 166 0.027261  0.011561
holdout_time catboost       raw        4   0.190   0.272    0.231 167 0.032188 -0.004479
holdout_time catboost    

In [7]:
# Pairwise permutation tests on |error| — for each split, compare all model pairs
# Use best-transform per model
sig_rows = []
perm_rng = np.random.default_rng(SEED)   # seeded once; each pair draws distinct permutations

for split_name, split_preds in all_preds.groupby('split'):
    # For each model, pick best transform (lowest RMSE)
    best_transform = (
        metrics_df[metrics_df['split'] == split_name]
        .sort_values('RMSE')
        .groupby('model', as_index=False)
        .first()[['model', 'transform']]
    )

    # Build dict model → {'y_true': ..., 'abs_err': ...}
    model_data = {}
    for _, row in best_transform.iterrows():
        grp = split_preds[
            (split_preds['model'] == row['model']) &
            (split_preds['target_transform'] == row['transform'])
        ]
        if len(grp) == 0:
            continue
        model_data[row['model']] = {
            'y_true':  grp['y_true'].values,
            'abs_err': np.abs(grp['y_pred'].values - grp['y_true'].values),
        }

    models_list = sorted(model_data.keys())
    print(f'\n[{split_name}] Pairwise tests for {len(models_list)} models ({len(list(combinations(models_list, 2)))} pairs) ...')

    for m_a, m_b in combinations(models_list, 2):
        yt_a = model_data[m_a]['y_true']
        yt_b = model_data[m_b]['y_true']
        e_a  = model_data[m_a]['abs_err']
        e_b  = model_data[m_b]['abs_err']

        # Verify sample correspondence before pairing
        if len(e_a) != len(e_b) or not np.array_equal(yt_a, yt_b):
            print(f'  WARNING: y_true mismatch between {m_a} and {m_b} — skipping pair')
            continue
        n = len(e_a)

        diff = e_a - e_b
        obs_stat = np.mean(diff)  # positive = A is worse

        # Permutation test: randomly flip signs of paired differences
        perm_stats = np.empty(N_PERM)
        for k in range(N_PERM):
            signs = perm_rng.choice([-1, 1], size=n)
            perm_stats[k] = np.mean(diff * signs)

        p_val = np.mean(np.abs(perm_stats) >= np.abs(obs_stat))

        sig_rows.append({
            'split': split_name,
            'model_a': m_a, 'model_b': m_b,
            'mean_ae_a': float(np.mean(e_a)),
            'mean_ae_b': float(np.mean(e_b)),
            'delta_mae': float(obs_stat),   # positive = A worse than B
            'p_value': float(p_val),
            'significant_05': p_val < 0.05,
        })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(SIG_DIR / 'pairwise_permutation.csv', index=False)
print(f'\nSaved: {SIG_DIR}/pairwise_permutation.csv')
sig_pairs = sig_df[sig_df['significant_05']].sort_values('p_value')
print(f'Significant pairs (p < 0.05): {len(sig_pairs)}')
if len(sig_pairs):
    print(sig_pairs[['split', 'model_a', 'model_b', 'delta_mae', 'p_value']].head(20).to_string(index=False))


[holdout_time] Pairwise tests for 13 models (78 pairs) ...

[loso] Pairwise tests for 11 models (55 pairs) ...

[stratified] Pairwise tests for 13 models (78 pairs) ...

Saved: D:\Regional-Scale-AOD-Estimation\modeling\results\significance/pairwise_permutation.csv
Significant pairs (p < 0.05): 154
       split    model_a       model_b  delta_mae  p_value
holdout_time   catboost           svr   0.005349      0.0
holdout_time   catboost  tabpfn_mi100   0.004705      0.0
holdout_time   catboost  tabpfn_mi200   0.006651      0.0
holdout_time   catboost  tabpfn_mi300   0.006992      0.0
holdout_time extratrees            rf  -0.001762      0.0
holdout_time   catboost tabpfn_pca150   0.006670      0.0
holdout_time extratrees  tabpfn_mi100   0.006069      0.0
holdout_time extratrees           svr   0.006713      0.0
holdout_time extratrees tabpfn_pca150   0.008034      0.0
holdout_time        gbm           svr   0.005992      0.0
holdout_time extratrees  tabpfn_mi200   0.008015      0.0
hold

In [8]:
# Significance heatmap — p-value matrix (models × models) per split
# Color tiers: p < 0.001 (dark green), p < 0.01 (green), p < 0.05 (yellow), ns (light grey)
# Cell text shows Δ MAE (positive = row model is worse than col model)

SIG_CMAP_BOUNDS = [0, 0.001, 0.01, 0.05, 1.01]
SIG_COLORS      = ['#1a7f37', '#2ea44f', '#f0c400', '#d0d7de']
sig_cmap        = mcolors.BoundaryNorm(SIG_CMAP_BOUNDS, len(SIG_COLORS))
sig_palette     = mcolors.ListedColormap(SIG_COLORS)

for split_name in sig_df['split'].unique():
    sub = sig_df[sig_df['split'] == split_name]
    models_uniq = sorted(set(sub['model_a']) | set(sub['model_b']))
    n = len(models_uniq)
    idx = {m: i for i, m in enumerate(models_uniq)}

    p_mat    = np.ones((n, n))    # p=1 on diagonal / unfilled
    delta_mat = np.full((n, n), np.nan)

    for _, row in sub.iterrows():
        i, j = idx[row['model_a']], idx[row['model_b']]
        p_mat[i, j]     = row['p_value']
        p_mat[j, i]     = row['p_value']
        delta_mat[i, j] =  row['delta_mae']   # positive = row worse
        delta_mat[j, i] = -row['delta_mae']   # negative = row better

    fig, ax = plt.subplots(figsize=(max(8, n * 0.75), max(7, n * 0.7)))
    ax.imshow(p_mat, cmap=sig_palette, norm=sig_cmap, aspect='auto')

    for i in range(n):
        for j in range(n):
            if i == j:
                ax.text(j, i, '—', ha='center', va='center', fontsize=7, color='grey')
            else:
                p_val = p_mat[i, j]
                d_val = delta_mat[i, j]
                stars = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
                d_str = f'{d_val:+.4f}' if not np.isnan(d_val) else ''
                ax.text(j, i, f'{stars}\n{d_str}',
                        ha='center', va='center', fontsize=6,
                        color='white' if p_val < 0.05 else '#24292f')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(models_uniq, rotation=40, ha='right', fontsize=8)
    ax.set_yticklabels(models_uniq, fontsize=8)
    ax.set_title(
        f'Pairwise Significance — {split_name}\n'
        'Δ MAE = row − col  (positive: row worse, negative: row better)\n'
        '★★★ p<0.001  ★★ p<0.01  ★ p<0.05  ns p≥0.05',
        fontsize=9
    )

    from matplotlib.patches import Patch
    legend_els = [
        Patch(color='#1a7f37', label='p < 0.001'),
        Patch(color='#2ea44f', label='p < 0.01'),
        Patch(color='#f0c400', label='p < 0.05'),
        Patch(color='#d0d7de', label='ns (p ≥ 0.05)'),
    ]
    ax.legend(handles=legend_els, loc='upper right',
              bbox_to_anchor=(1.28, 1.0), fontsize=8)

    plt.tight_layout()
    out_path = SIG_DIR / f'sig_heatmap_{split_name}.png'
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: sig_heatmap_{split_name}.png')

Saved: sig_heatmap_holdout_time.png
Saved: sig_heatmap_loso.png
Saved: sig_heatmap_stratified.png


In [9]:
# Scatter plots: one PNG per (split, model, transform)
# Saved to plots/scatter/{split}/{model}_{transform}.png

for split_name, split_preds in all_preds.groupby('split'):
    split_scatter_dir = SCATTER_DIR / split_name
    split_scatter_dir.mkdir(parents=True, exist_ok=True)

    for (model_name, transform), grp in split_preds.groupby(['model', 'target_transform']):
        yt = grp['y_true'].values
        yp = grp['y_pred'].values

        m_row = metrics_df[
            (metrics_df['split'] == split_name) &
            (metrics_df['model'] == model_name) &
            (metrics_df['transform'] == transform)
        ]
        if m_row.empty:
            continue
        m_row = m_row.iloc[0]

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(yt, yp, s=8, alpha=0.45, color='steelblue', rasterized=True)

        lim = max(yt.max(), yp.max()) * 1.06
        ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1 line')

        t_range = np.linspace(0, lim, 300)
        ax.fill_between(t_range,
                        t_range - (0.05 + 0.15 * t_range),
                        t_range + (0.05 + 0.15 * t_range),
                        alpha=0.15, color='orange', label='EE envelope')

        stats_text = (
            f'N = {int(m_row["N"])}\n'
            f'RMSE = {m_row["RMSE"]:.4f}\n'
            f'MAE  = {m_row["MAE"]:.4f}\n'
            f'MBE  = {m_row["MBE"]:+.4f}\n'
            f'r    = {m_row["Pearson_r"]:.3f}\n'
            f'R²   = {m_row["R2"]:.3f}\n'
            f'%EE  = {m_row["pct_within_EE"]:.1f}%'
        )
        ax.text(0.03, 0.97, stats_text,
                transform=ax.transAxes, fontsize=8,
                va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_xlabel('AOD true (550 nm)', fontsize=10)
        ax.set_ylabel('AOD predicted (550 nm)', fontsize=10)
        ax.set_title(f'{model_name} [{transform}] — {split_name}', fontsize=10)
        ax.legend(fontsize=8, loc='lower right')
        ax.set_aspect('equal', adjustable='box')

        out_path = split_scatter_dir / f'{model_name}_{transform}.png'
        fig.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

    n_files = len(list((SCATTER_DIR / split_name).glob('*.png')))
    print(f'Saved {n_files} scatter PNGs → plots/scatter/{split_name}/')

Saved 26 scatter PNGs → plots/scatter/holdout_time/
Saved 22 scatter PNGs → plots/scatter/loso/
Saved 26 scatter PNGs → plots/scatter/stratified/


In [10]:
# Bar chart: RMSE comparison across models and splits (best transform each)
plot_df = (
    metrics_df.sort_values('RMSE')
    .groupby(['split', 'model'], as_index=False)
    .first()
    .sort_values(['split', 'RMSE'])
)

splits = plot_df['split'].unique()
fig, axes = plt.subplots(1, len(splits), figsize=(7 * len(splits), 5), sharey=False)
if len(splits) == 1:
    axes = [axes]

for ax, split_name in zip(axes, splits):
    sub = plot_df[plot_df['split'] == split_name].sort_values('RMSE')
    # Clamp CI half-widths to zero — bootstrap can occasionally yield a percentile
    # above the point estimate on skewed distributions, making the bar go the wrong way.
    rmse_ci_lo = np.maximum(sub['RMSE'] - sub['RMSE_ci_lo'], 0)
    rmse_ci_hi = np.maximum(sub['RMSE_ci_hi'] - sub['RMSE'], 0)
    yerr = np.array([rmse_ci_lo.values, rmse_ci_hi.values])

    colors = ['#1f77b4' if t == 'raw' else '#ff7f0e' for t in sub['transform']]
    ax.bar(sub['model'], sub['RMSE'], color=colors,
           yerr=yerr, capsize=3, error_kw={'linewidth': 1})
    ax.set_title(f'{split_name}', fontsize=11)
    ax.set_ylabel('RMSE (AOD)' if ax is axes[0] else '')
    ax.set_xticklabels(sub['model'], rotation=35, ha='right', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.3f'))

from matplotlib.patches import Patch
legend_els = [Patch(color='#1f77b4', label='raw'), Patch(color='#ff7f0e', label='log1p')]
axes[-1].legend(handles=legend_els, fontsize=9, loc='upper left')

plt.suptitle('RMSE by Model and Split (95% bootstrap CI)', fontsize=12)
plt.tight_layout()
fig.savefig(PLOT_DIR / 'rmse_comparison.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Saved: rmse_comparison.png')

Saved: rmse_comparison.png


In [11]:
# Per-bin heatmaps: MAE (sequential) + MBE (diverging), all models, per split
# Rows = models sorted by overall RMSE, Cols = 5 AOD quintile bins
# Saved to per_bin/heatmap_mae_{split}.png and per_bin/heatmap_mbe_{split}.png

model_order_global = (
    metrics_df.sort_values('RMSE')
    .groupby(['split', 'model'], as_index=False).first()
)

for split_name in bin_df['split'].unique():
    # Best transform per model for this split
    best_tf = (
        model_order_global[model_order_global['split'] == split_name]
        .sort_values('RMSE')[['model', 'transform']]
        .reset_index(drop=True)
    )
    model_list   = best_tf['model'].tolist()
    tf_map       = best_tf.set_index('model')['transform'].to_dict()

    # Filter bin_df to best transform per model
    sub_bin = bin_df[bin_df['split'] == split_name].copy()
    sub_bin = sub_bin[
        sub_bin.apply(lambda r: r['transform'] == tf_map.get(r['model'], r['transform']), axis=1)
    ]

    # Build bin labels from actual quantile edges (same across models in a split)
    bin_labels = []
    for b in sorted(sub_bin['bin_idx'].unique()):
        grp_b = sub_bin[sub_bin['bin_idx'] == b]
        lo = grp_b['aod_lo'].iloc[0]
        hi = grp_b['aod_hi'].iloc[0]
        bin_labels.append(f'Q{b}\n{lo:.3f}–{hi:.3f}')

    n_models = len(model_list)
    n_bins   = len(bin_labels)

    mae_mat = np.full((n_models, n_bins), np.nan)
    mbe_mat = np.full((n_models, n_bins), np.nan)

    for mi, model_name in enumerate(model_list):
        for bi, b_idx in enumerate(sorted(sub_bin['bin_idx'].unique())):
            cell = sub_bin[(sub_bin['model'] == model_name) & (sub_bin['bin_idx'] == b_idx)]
            if not cell.empty:
                mae_mat[mi, bi] = cell['MAE'].values[0]
                mbe_mat[mi, bi] = cell['MBE'].values[0]

    def _draw_heatmap(mat, title, cmap, vmin, vmax, fmt, out_path, center=None):
        fig, ax = plt.subplots(figsize=(max(7, n_bins * 1.4), max(4, n_models * 0.5)))
        if center is not None:
            norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
            im = ax.imshow(mat, cmap=cmap, norm=norm, aspect='auto')
        else:
            im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

        for i in range(n_models):
            for j in range(n_bins):
                val = mat[i, j]
                if not np.isnan(val):
                    text_color = 'white' if abs(val) > (vmax - vmin) * 0.6 + vmin else 'black'
                    ax.text(j, i, fmt.format(val),
                            ha='center', va='center', fontsize=7, color='black')

        ax.set_xticks(range(n_bins))
        ax.set_xticklabels(bin_labels, fontsize=8)
        ax.set_yticks(range(n_models))
        ax.set_yticklabels(
            [f'{m} [{tf_map[m]}]' for m in model_list],
            fontsize=8
        )
        ax.set_xlabel('AOD quintile bin (true values)', fontsize=9)
        ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        plt.tight_layout()
        fig.savefig(out_path, dpi=120, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved: {out_path.name}')

    mae_max = np.nanmax(mae_mat)
    _draw_heatmap(
        mae_mat,
        title=f'Per-Bin MAE — {split_name}\n(lower = better; models sorted by overall RMSE)',
        cmap='YlOrRd',
        vmin=0, vmax=mae_max, center=None,
        fmt='{:.4f}',
        out_path=BIN_DIR / f'heatmap_mae_{split_name}.png'
    )

    mbe_abs_max = np.nanmax(np.abs(mbe_mat))
    _draw_heatmap(
        mbe_mat,
        title=f'Per-Bin MBE — {split_name}\n(blue = under-predict, red = over-predict; models sorted by overall RMSE)',
        cmap='RdBu_r',
        vmin=-mbe_abs_max, vmax=mbe_abs_max, center=0,
        fmt='{:+.4f}',
        out_path=BIN_DIR / f'heatmap_mbe_{split_name}.png'
    )

print('\nEvaluation complete.')

Saved: heatmap_mae_holdout_time.png
Saved: heatmap_mbe_holdout_time.png
Saved: heatmap_mae_loso.png
Saved: heatmap_mbe_loso.png
Saved: heatmap_mae_stratified.png
Saved: heatmap_mbe_stratified.png

Evaluation complete.
